# 06 · Experimento extra (post hoc): top-k = 5
**Status: experimento exploratório, NÃO substitui o placar oficial (notebook 05).** Ele foi planejado **depois** de ver os resultados do teste, que mostraram que a maioria das falhas vinha da busca (trecho certo fora do top-3, listas cortadas entre trechos). Por isso os resultados originais de B e C ficam intactos e este notebook salva em arquivos separados (`B_rag_k5`, `C_rag_lora_k5`).

Mudança única: `k=5` trechos no prompt (antes 3). Mesmo modelo, mesmo limiar (o gate usa o score do top-1, que não muda), mesmo adaptador LoRA (treinado com k=3: o ganho em C pode ser menor que em B por esse descasamento).

Com 20 perguntas no teste, 1 pergunta = 5 pontos: leia o IC95% antes de concluir qualquer coisa.

In [11]:
# --- Ambiente: funciona no Colab (via VSCode ou navegador) e localmente ---------------
import os, sys, subprocess
from pathlib import Path

REPO_URL = ""   # opcional: URL do repositório GitHub, se o projeto ainda não estiver na máquina do Colab

try:
    os.getcwd()
except FileNotFoundError:      # a pasta atual do kernel foi apagada/recriada: volta para um lugar válido
    os.chdir("/content" if Path("/content").exists() else str(Path.home()))

def achar_raiz():
    cwd = Path.cwd()
    cands = [cwd, *cwd.parents, Path("/content/GS2-GENIA"), Path("/content/drive/MyDrive/GS2-GENIA"),
             Path("/content/drive/MyDrive/GEN-IA/GS2-GENIA"), Path("/content/drive/MyDrive/GEN-IA")]
    for c in cands:
        if (c / "src" / "config.py").exists():
            return c
    for base in (Path("/content"), Path("/content/drive/MyDrive")):   # busca 1-2 níveis abaixo
        if base.exists():
            for f in list(base.glob("*/src/config.py")) + list(base.glob("*/*/src/config.py")):
                return f.parent.parent
    return None

ROOT = achar_raiz()
if ROOT is None and REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, "/content/GS2-GENIA"], check=True)
    ROOT = Path("/content/GS2-GENIA")
assert ROOT is not None, "Não achei src/config.py: abra o notebook dentro da pasta GS2-GENIA ou preencha REPO_URL."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

ROOT = /content/drive/MyDrive/GS2-GENIA


In [12]:
%pip install -q -r requirements.txt

In [13]:
import json, gc, torch, pandas as pd
from src import config
from src.ingest import load_chunks
from src.evaluate import (load_testset, run_config, aggregate, scoreboard, bootstrap_diff)
from src.rag import Generator, answer
from src.retrieval import Retriever, load_threshold, evaluate_recall_fatos

K = 5
chunks = load_chunks(); by_id = {c["chunk_id"]: c for c in chunks}
retriever = Retriever(chunks)
TH = load_threshold(); items = load_testset()
print("limiar =", TH, "| k =", K)
todas = items
print("fatos da resposta presentes no top-3:", round(evaluate_recall_fatos(retriever, todas, k=3), 3),
      "| no top-5:", round(evaluate_recall_fatos(retriever, todas, k=K), 3))

W1116 03:41:42.053000 108462 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1116 03:41:42.098000 108462 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

limiar = 0.8358593756158148 | k = 5
fatos da resposta presentes no top-3: 0.75 | no top-5: 0.893


## B com k=5 (modelo base + RAG simples)

In [14]:
gen = Generator()
rows_Bk = run_config("B_rag_k5", items, lambda q: answer(q, gen, retriever, mode="rag", threshold=TH, k=K, chunks_by_id=by_id))
for sp in ("dev", "teste"):
    print("B k=5", sp, aggregate([r for r in rows_Bk if r["split"] == sp]))
gen = None; gc.collect(); torch.cuda.empty_cache()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

B_rag_k5:   0%|          | 0/40 [00:00<?, ?it/s]

B k=5 dev {'n': 20, 'acuracia_%': 60.0, 'acerto_respondiveis_%': 42.9, 'recusa_correta_%': 100.0, 'alucinacao_%': 35.0, 'recusa_indevida_%': 0.0, 'json_valido_%': 95.0, 'citacao_correta_%': 53.8, 'taxa_resposta_%': 65.0, 'erro_entre_respondidas_%': 53.8}
B k=5 teste {'n': 20, 'acuracia_%': 50.0, 'acerto_respondiveis_%': 57.1, 'recusa_correta_%': 33.3, 'alucinacao_%': 25.0, 'recusa_indevida_%': 0.0, 'json_valido_%': 75.0, 'citacao_correta_%': 76.9, 'taxa_resposta_%': 65.0, 'erro_entre_respondidas_%': 38.5}


## C com k=5 (RAG + LoRA)

In [15]:
genC = Generator(adapter_path=config.ADAPTER_DIR)
rows_Ck = run_config("C_rag_lora_k5", items, lambda q: answer(q, genC, retriever, mode="rag", threshold=TH, k=K, chunks_by_id=by_id))
for sp in ("dev", "teste"):
    print("C k=5", sp, aggregate([r for r in rows_Ck if r["split"] == sp]))
genC = None; gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

C_rag_lora_k5:   0%|          | 0/40 [00:00<?, ?it/s]

C k=5 dev {'n': 20, 'acuracia_%': 65.0, 'acerto_respondiveis_%': 50.0, 'recusa_correta_%': 100.0, 'alucinacao_%': 30.0, 'recusa_indevida_%': 7.1, 'json_valido_%': 100.0, 'citacao_correta_%': 46.2, 'taxa_resposta_%': 65.0, 'erro_entre_respondidas_%': 46.2}
C k=5 teste {'n': 20, 'acuracia_%': 70.0, 'acerto_respondiveis_%': 57.1, 'recusa_correta_%': 100.0, 'alucinacao_%': 30.0, 'recusa_indevida_%': 0.0, 'json_valido_%': 100.0, 'citacao_correta_%': 64.3, 'taxa_resposta_%': 70.0, 'erro_entre_respondidas_%': 42.9}


## Comparação com o placar oficial (k=3)

In [16]:
def carrega(nome):
    return [json.loads(l) for l in open(config.RESULTS_DIR / f"{nome}.jsonl", encoding="utf-8")]
R = {"B (k=3)": carrega("B_rag"), "B (k=5)": carrega("B_rag_k5"),
     "C (k=3)": carrega("C_rag_lora"), "C (k=5)": carrega("C_rag_lora_k5")}
teste = {k: [r for r in v if r["split"] == "teste"] for k, v in R.items()}
dev = {k: [r for r in v if r["split"] == "dev"] for k, v in R.items()}
print("== TESTE =="); display(scoreboard(teste))
print("== DEV =="); display(scoreboard(dev))
(ROOT / "avaliacao").mkdir(exist_ok=True)
scoreboard(teste).to_csv(ROOT / "avaliacao" / "placar_k5_teste.csv"); scoreboard(dev).to_csv(ROOT / "avaliacao" / "placar_k5_dev.csv")

for nome, x, y in [("B k5 - B k3", teste["B (k=3)"], teste["B (k=5)"]), ("C k5 - C k3", teste["C (k=3)"], teste["C (k=5)"])]:
    for chave in ("correta", "alucinacao"):
        r = bootstrap_diff(x, y, key=chave)
        print(f"{nome} [{chave}]: diferença={r['diff']:+.3f}  IC95%=({r['ic95'][0]:+.3f}, {r['ic95'][1]:+.3f})  n={r['n']}")

== TESTE ==


,n,acuracia_%,acerto_respondiveis_%,recusa_correta_%,alucinacao_%,recusa_indevida_%,json_valido_%,citacao_correta_%,taxa_resposta_%,erro_entre_respondidas_%
B (k=3),20.0,35.0,35.7,33.3,55.0,0.0,90.0,64.3,80.0,68.8
B (k=5),20.0,50.0,57.1,33.3,25.0,0.0,75.0,76.9,65.0,38.5
C (k=3),20.0,55.0,35.7,100.0,40.0,7.1,100.0,53.8,65.0,61.5
C (k=5),20.0,70.0,57.1,100.0,30.0,0.0,100.0,64.3,70.0,42.9


== DEV ==


,n,acuracia_%,acerto_respondiveis_%,recusa_correta_%,alucinacao_%,recusa_indevida_%,json_valido_%,citacao_correta_%,taxa_resposta_%,erro_entre_respondidas_%
B (k=3),20.0,55.0,42.9,83.3,35.0,0.0,90.0,69.2,65.0,53.8
B (k=5),20.0,60.0,42.9,100.0,35.0,0.0,95.0,53.8,65.0,53.8
C (k=3),20.0,60.0,42.9,100.0,35.0,7.1,100.0,69.2,65.0,53.8
C (k=5),20.0,65.0,50.0,100.0,30.0,7.1,100.0,46.2,65.0,46.2


B k5 - B k3 [correta]: diferença=+0.150  IC95%=(+0.000, +0.300)  n=20
B k5 - B k3 [alucinacao]: diferença=-0.300  IC95%=(-0.500, -0.100)  n=20
C k5 - C k3 [correta]: diferença=+0.150  IC95%=(+0.000, +0.300)  n=20
C k5 - C k3 [alucinacao]: diferença=-0.100  IC95%=(-0.250, +0.000)  n=20


In [17]:
# quais perguntas mudaram de resultado com k=5 (todas as 40)
a = {r["id"]: r for r in R["C (k=3)"]}; b = {r["id"]: r for r in R["C (k=5)"]}
for i in sorted(a):
    if a[i]["correta"] != b[i]["correta"]:
        print(i, "C k3:", "certa" if a[i]["correta"] else "errada", "->", "C k5:", "certa" if b[i]["correta"] else "errada",
              "|", a[i]["pergunta"][:70])

Q08 C k3: errada -> C k5: certa | Qual é a função do Comitê Nacional de Manejo Integrado do Fogo (Comif)
Q24 C k3: errada -> C k5: certa | Segundo o SCI-100, como e por que surgiu o SCI nos Estados Unidos?
Q25 C k3: errada -> C k5: certa | Segundo o SCI-100, qual é a diferença entre comando individual e coman
Q28 C k3: errada -> C k5: certa | Qual a periodicidade recomendada para simulados completos de abandono 


## Como reportar
- Se C (k=5) ganhar acima do IC95%: reportar como *experimento adicional post hoc*, com o placar oficial intacto.
- Se não ganhar (ou piorar por descasamento com o treino em k=3): reportar igualmente — é um resultado negativo útil (a busca precisa de outra mudança, p.ex. híbrida BM25, ou o LoRA de ser retreinado com k=5).
- Nunca substitua os números do notebook 05 por estes.